### TfIdf with feature selection

Inspired by [tf-idf-spark-and-python](https://github.com/logicalguess/tf-idf-spark-and-python/tree/master)

In [80]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath("../src"))

In [81]:
labeled_df = pd.read_csv('../data/labeledTrainData.tsv.zip', header=0, delimiter="\t", quoting=3)
unlabeled_df = pd.read_csv('../data/unlabeledTrainData.tsv.zip', header=0, delimiter="\t", quoting=3)

print(f"labeled_df shape: {labeled_df.shape}")
print(f"unlabeled_df shape: {unlabeled_df.shape}")

labeled_df shape: (25000, 3)
unlabeled_df shape: (50000, 2)


In [82]:
import my_utils # noqa: F401

wordlist_labeled_train = labeled_df.review.nlp.review_to_wordlist()
wordlist_unlabeled_train = unlabeled_df.review.nlp.review_to_wordlist()

print(f"wordlist_labeled_train shape: {wordlist_labeled_train.shape}")
print(f"wordlist_unlabeled_train shape: {wordlist_unlabeled_train.shape}")

wordlist_all =  pd.concat([wordlist_labeled_train, wordlist_unlabeled_train], ignore_index=True)
corpus_all = [' '.join(w) for w in wordlist_all.tolist()]
corpus_train = [' '.join(w) for w in wordlist_labeled_train.tolist()]

print(f"Total number of reviews in corpus_all: {len(corpus_all)}")
print(f"Total number of reviews in corpus_train: {len(corpus_train)}")

wordlist_labeled_train shape: (25000,)
wordlist_unlabeled_train shape: (50000,)
Total number of reviews in corpus_all: 75000
Total number of reviews in corpus_train: 25000


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english', max_df=0.85)
vectorizer = vectorizer.fit(corpus_all)
tfidf_matrix = vectorizer.transform(corpus_train)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

Vocabulary size: 123129
TF-IDF matrix shape: (25000, 123129)


In [84]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2


fselect = SelectKBest(chi2, k=7000)
train_data_features = fselect.fit_transform(tfidf_matrix, labeled_df.sentiment)
train_data_features.shape


X_train, X_test, y_train, y_test = train_test_split(
    train_data_features, labeled_df.sentiment, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((20000, 7000), (5000, 7000), (20000,), (5000,))

In [85]:
from sklearn.svm import LinearSVC


clf = LinearSVC(random_state=42, max_iter=1000)
clf.fit(X_train, y_train)
score = clf.score(X_test, y_test)
print(f'LinearSVC sentiment score: {score}')

LinearSVC sentiment score: 0.9032
